In [3]:
import numpy as np

def calc_power(fluid):
    """
    calculate fluid power delivered. Pump efficiency not considered.
    """
   
    # system parameters
    Q = 3.42   # Heat load from PTR (W)
    delT = 30  # temperature change of fluid (K)
    
    L = 20     # total pipe length (m)
    D = 0.03   # Inner Pipe Diameter (m)

    # fluid parameters
    cp    = fluid['cp']     # heat capacity  J/(kg K)
    rho_f = fluid['rho_f']  # fluid density  kg/m3
    mu    = fluid['mu']     # dynamic viscosity (Pa s)

    # flow calculations
    mt = Q / (cp*delT)       # mass flow rate
    Vt = mt / rho_f          # volumetric flow rate
    Ac = 0.25*np.pi*D*D      # pipe inner cross section area
    v = mt / (rho_f*Ac)      # flow speed
    Re = rho_f * v * D / mu  # Reynolds Number
    
    # find friction factor
    if Re < 2300:
        f = 64/Re
    elif Re >= 4000:
        f = (0.79*np.log(Re)-1.64)**(-2)
    else:
        print("undefined Re.")
    
    # pressure loss 
    delP_pipe = f * (L / D) * (0.5*rho_f*v*v)   # pressure loss from friction in the straight pipe
    delP_comp = 0.                                # pressure loss from components
    delP_tot = delP_pipe + delP_comp
    
    P_f = delP_tot * Vt      # power needed to be delivered to fluid

    # fluid mass
    mass_f = Ac * L * rho_f
    
    return P_f, mass_f

def mass_frac_from_volume_frac(volume_frac):
    """
    calculate mass fraction of Ethylene Glycol from its volume fraction
    """
    rho_eg = 1113.
    rho_h2o = 1000.
    return rho_eg/rho_h2o*volume_frac

In [5]:
# Thermodynamics properties of fluids

# CFC 11  (Reference temperature: 242.5 K)
CFC11 = {
    'cp': 840.38,            # J/(kg K)    # Osborne, The Heat Capacity, Entropy, Heats of Fusion 
                                     # and Vaporization and Vapor Pressure of Fluorotrichloromethane
    'rho_f': 1601.9,         # kg/m3       # NIST https://webbook.nist.gov/cgi/fluid.cgi?ID=C75694&Action=Page
    'mu': 7.8308 * 1e-4      # (Pa s)      # NIST https://webbook.nist.gov/cgi/fluid.cgi?ID=C75694&Action=Page
}

# Therminol®59  (Reference temperature: 223.15 K)
Therminol59 = {
    'cp': 1460,              # J/(kg K)    # https://domxoloda.ru/oils/docs/HTF-59.PDF
    'rho_f': 1025,           # kg/m3       # https://domxoloda.ru/oils/docs/HTF-59.PDF
    'mu': 25043.1 * 1e-4     # (Pa s)      # https://domxoloda.ru/oils/docs/HTF-59.PDF
}

# Therminol®LT  (Reference temperature: 223.15 K)
TherminolLT = {
    'cp': 1530,              # J/(kg K)    # https://www.sintelub.com/wp-content/uploads/PDS/29.Therminol_LT.pdf
    'rho_f': 921,            # kg/m3       # https://www.sintelub.com/wp-content/uploads/PDS/29.Therminol_LT.pdf
    'mu': 39.9e-4,           # (Pa s)      # https://www.sintelub.com/wp-content/uploads/PDS/29.Therminol_LT.pdf
    'nu': 4.33,              # mm2/s       # https://www.sintelub.com/wp-content/uploads/PDS/29.Therminol_LT.pdf
    'k': 0.1384              # W/(m K)     # https://www.sintelub.com/wp-content/uploads/PDS/29.Therminol_LT.pdf
}

# Syltherm®XLT  (Reference temperature: 242.5 K)
SylthermXLT = {
    'cp': 1666,              # J/(kg K)    # https://www.npl.washington.edu/TRIMS/sites/sand.npl.washington.edu.TRIMS/files/manuals-documentation/syltherm-xlt-technical-data-sheet.pdf
    'rho_f': 907.2,          # kg/m3       # https://www.npl.washington.edu/TRIMS/sites/sand.npl.washington.edu.TRIMS/files/manuals-documentation/syltherm-xlt-technical-data-sheet.pdf
    'mu': 130 * 1e-4,        # (Pa s)      # https://www.npl.washington.edu/TRIMS/sites/sand.npl.washington.edu.TRIMS/files/manuals-documentation/syltherm-xlt-technical-data-sheet.pdf
    'k': 0.1212              # W/(m K)     # https://www.npl.washington.edu/TRIMS/sites/sand.npl.washington.edu.TRIMS/files/manuals-documentation/syltherm-xlt-technical-data-sheet.pdf
}

# Therminol®VLT  (Reference temperature: 222.15 K)
TherminolVLT = {
    'cp': 1650,              # J/(kg K)    # https://www.sintelub.com/wp-content/uploads/PDS/30.Therminol_VLT.pdf
    'rho_f': 809,            # kg/m3       # https://www.sintelub.com/wp-content/uploads/PDS/30.Therminol_VLT.pdf
    'mu': 23.9 * 1e-4,        # (Pa s)      # https://www.sintelub.com/wp-content/uploads/PDS/30.Therminol_VLT.pdf
    'k': 0.1181              # W/(m K)     # https://www.sintelub.com/wp-content/uploads/PDS/30.Therminol_VLT.pdf
}

# R-236fa  (Reference temperature: 242.5 K)
R_236fa = {
    'cp': 1166.1,            # J/(kg K)    # NIST https://webbook.nist.gov/cgi/cbook.cgi?ID=C690391&Units=SI
    'rho_f': 1533.9,         # kg/m3       # NIST https://webbook.nist.gov/cgi/cbook.cgi?ID=C690391&Units=SI
    'mu': 6.1904e-4          # (Pa s)      # NIST https://webbook.nist.gov/cgi/cbook.cgi?ID=C690391&Units=SI
}

# Ethylene Glycol based Water Solutions (223.15 K, 50%.Vol)
Ethylene_Glycol = {
    'cp': 3040,            # J/(kg K)    # https://www.engineeringtoolbox.com/ethylene-glycol-d_146.html
    'rho_f': 1127,         # kg/m3       # https://www.engineeringtoolbox.com/ethylene-glycol-d_146.html
    'mu': 0.0644           # (Pa s)      # https://www.mokon.com/products/fluids/glycol-solutions/pdf/Ethylene-Glycol-Technical-Data.pdf
}

In [3]:
calc_power(CFC11)

(5.649352699934253e-09, 22.6463277230347)

In [4]:
calc_power(Therminol59)

(1.4620068981908622e-05, 14.49059611468292)

In [5]:
calc_power(R_236fa)

(2.5296929079175134e-09, 21.685000371036224)

In [6]:
calc_power(SylthermXLT)

(7.440498808568438e-08, 12.82523784901497)

In [7]:
calc_power(TherminolLT)

(2.6271541138340377e-08, 13.020330752802897)

In [8]:
calc_power(TherminolVLT)

(1.75366948771482e-08, 11.43696805539364)

In [6]:
calc_power(Ethylene_Glycol)

(7.173079504430434e-08, 15.932587142680633)